# Строгий датасет: Сфера → CDI → ЕГРН

## Что делает ноутбук

Ноутбук собирает строгий датасет по объектам недвижимости и дополняет его сведениями из CDI и ЕГРН. Одна строка итоговой таблицы — один объект Сферы в одном договоре. Если объект не удалось однозначно связать с ЕГРН, строка всё равно остаётся в датасете, а поля ЕГРН остаются пустыми.

Полный путь данных:

```text
Сфера: договор → заявка → задача оформления → объект недвижимости
CDI: CDI ID страхователя → адрес застрахованного объекта → ФИАС дома
ЕГРН: ФИАС дома → здание → сведения ЕГРН
```

## Как работает соединение

### 1. Объекты Сферы

Из Сферы выбираются объекты недвижимости, у которых есть официальная связь с договором через заявку и задачу оформления. Из Сферы также берутся CDI ID и ИНН страхователя, адрес объекта, площадь и страховая сумма.

### 2. Адрес в CDI

CDI ID относится к страхователю, а не к зданию. Сначала по CDI ID находится клиент, затем его действующий адрес с признаком адреса застрахованного объекта. Юридический адрес компании для этой связи не используется.

ИНН из Сферы сравнивается с ИНН найденного клиента CDI:

- `confirmed` — ИНН совпали;
- `conflict` — ИНН не совпали, адрес CDI не используется;
- `not_checked` — в одном из источников нет данных для проверки.

Адрес CDI принимается только тогда, когда найден один клиент и один действующий адрес застрахованного объекта. Если адресов несколько, ноутбук не выбирает один из них случайно.

### 3. ФИАС дома

Из выбранного адреса CDI берётся ФИАС дома. Функция CDI приводит код к уровню дома и возвращает координаты. Если ФИАС получить не удалось, поиск в ЕГРН для этой строки не выполняется.

### 4. ЕГРН

По ФИАС дома в ЕГРН ищутся здания, строения и сооружения. Помещения, квартиры и офисы в этом варианте не присоединяются. Повторные версии одной записи ЕГРН удаляются.

Решение принимается так:

- найден один объект ЕГРН — связь принимается, метод `fias_house_only`;
- найдено несколько объектов и площадь Сферы заполнена — кандидаты сравниваются по площади;
- после сравнения по площади остался один объект — связь принимается, метод `fias_house_and_area`;
- площади нет и найдено несколько объектов — связь не устанавливается, статус `ambiguous`;
- площадь есть, но после сравнения осталось несколько объектов — связь не устанавливается, статус `ambiguous`;
- по ФИАС ничего не найдено — статус `not_found`.

Площадь используется только для уточнения среди нескольких кандидатов. Если по ФИАС уже найден один объект, отсутствие площади не мешает соединению.

## Что сохраняется

Ноутбук сохраняет итоговый датасет и отдельные снимки кандидатов CDI, результатов функции ФИАС и поиска в ЕГРН. Количество строк итогового датасета должно совпадать с количеством строк строгого датасета из Сферы.

КХД используется только на чтение. Временные таблицы не создаются.


In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

In [ ]:
import json
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

# 1. Подключение к Сфере


In [ ]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')


In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

# 2. Подключение к Oracle КХД



In [ ]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')
CDI_FUNCTION_SCHEMA = credentials.get('CDI_FUNCTION_SCHEMA', 'DM_MOTOR')
CDI_FUNCTION_NAME = credentials.get(
    'CDI_FUNCTION_NAME',
    'F_GET_CDI_ADDR_BY_CODE',
)


In [ ]:
required_khd_tables = {
    'EGRN_DATA',
    'STG_ADDRESS_CDI_ZUD',
    'STG_PARTY_SRC',
    'STG_LEGAL_CLIENTS',
}

with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name in (
              'EGRN_DATA',
              'STG_ADDRESS_CDI_ZUD',
              'STG_PARTY_SRC',
              'STG_LEGAL_CLIENTS'
          )
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = {row[0] for row in cursor.fetchall()}

    cursor.execute(
        """
        select count(*)
        from all_procedures
        where owner = :owner
          and object_name = :object_name
        """,
        owner=CDI_FUNCTION_SCHEMA.upper(),
        object_name=CDI_FUNCTION_NAME.upper(),
    )
    cdi_function_visible = cursor.fetchone()[0] > 0

missing_khd_tables = sorted(required_khd_tables - available_khd_tables)
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Таблицы CDI и ЕГРН доступны')
print('Функция нормализации CDI видна:', cdi_function_visible)


# 3. SQL Сфера, строгий подход

In [ ]:
strict_sql = r"""
/*
Для чего нужен запрос
---------------------
Запрос собирает основу датасета для модели 1 по недвижимости ЮЛ.
Он объединяет сведения о договоре, объекте, адресе, страхователе,
отрасли, страховых суммах и ближайшем предыдущем договоре.

Одна строка результата
----------------------
Одна строка - один объект недвижимости в одном договоре.
Один договор может занимать несколько строк, если в нем несколько объектов.


Как связаны таблицы
-------------------
Договор -> заявка -> задача оформления -> объект в задаче
        -> характеристики объекта -> сам объект -> адрес
        -> условия страхования объекта

Из договора берется страхователь. Из заявки берется CRM-карточка,
в которой находятся отрасль и сегмент.

Какие записи попадают в результат
---------------------------------
- задача оформления договора: task_type = draft_contract;
- завершенная рабочая задача: status = operational_archive;
- тип документа: ins_document_type = new_ins_contract,
  ins_contract_prolong или NULL, то есть новый договор, пролонгация
  или незаполненное значение;
- отказ в страховании не установлен: ins_refuse IS NOT TRUE;
- дата удаления отсутствует: d_delete IS NULL для задачи, заявки,
  договора и объекта;
- тип объекта: elementary_obj_type = nedv_ul_and_ip,
  то есть недвижимость ЮЛ и ИП.


Как читать страховые суммы
--------------------------
- contract_insured_sum - общая СС всего договора;
- task_object_insured_sum - СС объекта в строке связи задачи и объекта;
- condition_min_insured_sum и condition_max_insured_sum - минимальная и
  максимальная СС среди условий выбранной версии объекта;
- insured_sum - СС среди условий выбранной версии объекта.


Как используется история
-------------------------
Ближайший предыдущий договор ищется по bps_contract.prevcontract_id.
Прошлая СС объекта заполняется только тогда, когда в текущем и предыдущем
договоре совпал object_id. Если при пролонгации объект завели с новым ID,
прошлая объектная СС останется пустой.

*/

with task_candidates as (
    /* Шаг 1. Находим все подходящие задачи оформления. */
    select
        c.id as contract_id,
        c.n_contract as contract_number,
        c.prevcontract_id as previous_contract_id,
        c.rootcontract_id as root_contract_id,
        c.contractor_id as policyholder_id,
        c.document_status as contract_status,
        c.d_sign_contract as contract_sign_date,
        c.d_start_contract as contract_start_date,
        c.d_end_contract as contract_end_date,
        c.currency as contract_currency,
        c.ins_product_sbs as insurance_product,
        c.ins_program as insurance_program,

        r.id as request_id,
        r.corporate_crm_id,
        r.business_segment,

        t.id as task_id,
        t.d_create as task_create_date,
        t.d_change as task_change_date,
        t.task_type,
        t.status as task_status,
        t.ins_document_type,
        t.contract_type,
        t.d_conclusion_ins_contract as contract_conclusion_date,
        t.ins_refuse,
        t.industry as task_industry,
        t.subindustry as task_subindustry,
        t.total_ins_contract_amount as contract_insured_sum,
        t.total_ins_contract_premium as contract_premium,
        t.curr_ins_contract_amount as contract_amount_currency,

        coalesce(
            t.d_conclusion_ins_contract::timestamp with time zone,
            c.d_sign_contract,
            t.d_create
        ) as as_of_date,

        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* Шаг 2. Для каждого договора оставляем одну самую позднюю задачу. */
    select *
    from task_candidates
    where task_number = 1
),

contract_context as (
    /* Шаг 3. Добавляем ближайший предыдущий договор, если он указан. */
    select
        current_task.*,
        previous_contract.n_contract as previous_contract_number,
        previous_contract.d_start_contract as previous_contract_start_date,
        previous_contract.d_end_contract as previous_contract_end_date,
        previous_task.contract_insured_sum as previous_contract_insured_sum,
        previous_task.contract_premium as previous_contract_premium,
        previous_task.contract_amount_currency
            as previous_contract_amount_currency
    from selected_tasks current_task
    left join bps_contract previous_contract
        on previous_contract.id = current_task.previous_contract_id
    left join selected_tasks previous_task
        on previous_task.contract_id = current_task.previous_contract_id
),

object_candidates as (
    /*
    Шаг 4. К выбранной задаче присоединяем объекты недвижимости.
    Здесь же добавляем адрес, страхователя, CRM и характеристики объекта.
    */
    select
        contract.*,

        policyholder.inn as policyholder_inn,
        policyholder.contractor_type as policyholder_type,
        policyholder.cdi_id as policyholder_cdi_id,
        policyholder.ogrn as policyholder_ogrn,
        policyholder.kpp as policyholder_kpp,
        policyholder.company_name_short as policyholder_name,
        policyholder.company_form as policyholder_company_form,
        policyholder.company_register_day
            as policyholder_registration_date,

        crm.id as crm_id,
        crm.client_id as crm_client_id,
        crm.segment as crm_segment,
        crm.macroindustry as crm_macroindustry,
        crm.industry as crm_industry,
        crm.primary_occupation as crm_primary_occupation,
        crm.specialization as crm_specialization,
        crm.okved as crm_okved,

        link.id as task_object_link_id,
        link.characteristics_id,
        link.object_group_id,
        link.insured_sum as task_object_insured_sum,
        link.insured_sum_currency as task_object_insured_sum_currency,
        link.per_occurance_limit as task_object_per_occurrence_limit,

        obj.id as object_id,
        obj.obj_name as object_name,
        obj.description as object_description,
        obj.obj_type as object_type,
        obj.elementary_obj_type,
        obj.original_address,
        obj.geo_address_id,

        ch.version_number as characteristics_version_number,
        ch.version_start_date as characteristics_version_start_date,
        ch.version_end_date as characteristics_version_end_date,
        ch.version_is_active as characteristics_version_is_active,
        ch.insurance_value,
        ch.insurance_value_currency,
        ch.insurance_value_basis,
        ch.is_pledged,
        ch.pledged_value,
        ch.ownership_type,
        ch.is_leased,
        ch.insured_components,
        ch.activity_types,
        ch.risk_natures,
        ch.insurance_territory,
        ch.characteristics ->> 'total_area_sq_m' as total_area,
        ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
        ch.characteristics ->> 'construction_year' as construction_year,
        ch.characteristics ->> 'last_capital_repair_year'
            as capital_repair_year,
        ch.characteristics ->> 'total_floors_count' as floors_count,
        ch.characteristics ->> 'occupied_floor' as occupied_floor,
        ch.characteristics ->> 'load_bearing_walls_material'
            as walls_material,
        ch.characteristics ->> 'interfloor_overlap_material'
            as overlap_material,
        ch.characteristics ->> 'roofing_material' as roofing_material,
        ch.characteristics as object_characteristics_json,

        address.full_address,
        address.postal_code,
        address.region_id as address_region_id,
        address.area as district,
        address.settlement,
        address.street,
        address.house,
        address.building,
        address.block,
        address.flat,
        address.office,
        address.fias_code,
        address.longitude,
        address.latitude,
        address.address_dgis_id,

        row_number() over (
            partition by contract.task_id, obj.id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc,
                ch.id desc
        ) as object_number
    from contract_context contract
    join bps_request_ins_task_insurance_object link
        on link.parent_id = contract.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    left join base_geo_address address
        on address.id = obj.geo_address_id
    left join bps_contractor policyholder
        on policyholder.id = contract.policyholder_id
    left join bps_corporate_crm crm
        on crm.id = contract.corporate_crm_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_objects as (
    /*
    Шаг 5. Если объект несколько раз связан с одной задачей,
    оставляем одну самую позднюю запись связи.
    */
    select *
    from object_candidates
    where object_number = 1
),

selected_characteristics as (
    /* Шаг 6. Получаем список версий объектов для поиска их условий. */
    select distinct characteristics_id
    from selected_objects
),

condition_summary as (
    /*
    Шаг 7. У одной версии объекта может быть несколько вариантов условий.
    Сворачиваем их в одну строку, чтобы один объект не продублировался.
    */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        min(cond.insured_sum) as condition_min_insured_sum,
        max(cond.insured_sum) as condition_max_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as condition_currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        jsonb_agg(
            jsonb_strip_nulls(
                jsonb_build_object(
                    'option_number', cond.terms_option_number,
                    'insured_sum', cond.insured_sum,
                    'currency', cond.insured_sum_currency,
                    'per_occurrence_limit', cond.per_occurance_limit
                )
            )
            order by cond.terms_option_number nulls last, cond.id
        ) as conditions_json
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

object_data as (
    /* Шаг 8. Добавляем к каждому объекту найденные суммы и условия. */
    select
        obj.*,
        conditions.condition_count,
        conditions.condition_min_insured_sum,
        conditions.condition_max_insured_sum,
        conditions.condition_currency_count,
        conditions.insured_sum_currency,
        conditions.minimum_per_occurrence_limit,
        conditions.maximum_per_occurrence_limit,
        conditions.conditions_json
    from selected_objects obj
    left join condition_summary conditions
        on conditions.characteristics_id = obj.characteristics_id
),

objects_with_previous as (
    /*
    Шаг 9. Ищем тот же object_id в ближайшем предыдущем договоре
    и, если нашли, добавляем его предыдущую СС.
    */
    select
        current_object.*,
        case
            when previous_object.condition_min_insured_sum =
                 previous_object.condition_max_insured_sum
             and previous_object.condition_currency_count <= 1
            then previous_object.condition_max_insured_sum
        end as previous_object_insured_sum,
        previous_object.insured_sum_currency
            as previous_object_insured_sum_currency
    from object_data current_object
    left join object_data previous_object
        on previous_object.contract_id = current_object.previous_contract_id
       and previous_object.object_id = current_object.object_id
),

raw_result as (
/* Шаг 10. Собираем исходные поля строгого датасета. */
select
    /* Основные ID. */
    obj.contract_id,
    obj.contract_number,
    obj.previous_contract_id,
    obj.root_contract_id,
    obj.request_id,
    obj.task_id,
    obj.task_object_link_id,
    obj.characteristics_id,
    obj.object_id,
    obj.geo_address_id,
    obj.policyholder_id,
    obj.corporate_crm_id,

    /* Договор и его даты. */
    obj.as_of_date,
    obj.contract_conclusion_date,
    obj.contract_sign_date,
    obj.contract_start_date,
    obj.contract_end_date,
    obj.contract_status,
    obj.ins_document_type,
    obj.contract_type,
    obj.contract_currency,
    obj.insurance_product,
    obj.insurance_program,

    /* Объект. */
    count(*) over (
        partition by obj.contract_id
    ) as real_estate_objects_in_contract,
    obj.object_group_id,
    obj.object_name,
    obj.object_description,
    obj.object_type,
    obj.elementary_obj_type,
    obj.total_area,
    obj.occupied_area,
    obj.construction_year,
    obj.capital_repair_year,
    obj.floors_count,
    obj.occupied_floor,
    obj.walls_material,
    obj.overlap_material,
    obj.roofing_material,
    obj.ownership_type,
    obj.is_leased,
    obj.insured_components,
    obj.activity_types,
    obj.risk_natures,
    obj.insurance_territory,

    /* Адрес. */
    obj.full_address,
    obj.original_address,
    obj.postal_code,
    obj.address_region_id,
    obj.district,
    obj.settlement,
    obj.street,
    obj.house,
    obj.building,
    obj.block,
    obj.flat,
    obj.office,
    obj.fias_code,
    obj.longitude,
    obj.latitude,
    obj.address_dgis_id,

    /* Страхователь, отрасль и сегмент. */
    obj.policyholder_inn,
    obj.policyholder_type,
    obj.policyholder_cdi_id,
    obj.policyholder_ogrn,
    obj.policyholder_kpp,
    obj.policyholder_name,
    obj.policyholder_company_form,
    obj.policyholder_registration_date,
    obj.crm_id,
    obj.crm_client_id,
    (obj.crm_client_id = obj.policyholder_id) as crm_client_is_policyholder,
    obj.crm_segment,
    obj.crm_macroindustry,
    obj.crm_industry,
    obj.crm_primary_occupation,
    obj.crm_specialization,
    obj.crm_okved,
    obj.business_segment,
    obj.task_industry,
    obj.task_subindustry,

    /*
    Все СС стоят рядом.
    СС договора относится ко всему договору и повторяется у его объектов.
    */
    obj.contract_insured_sum,
    obj.contract_amount_currency,
    min(obj.condition_min_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_min_insured_sum,
    max(obj.condition_max_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_max_insured_sum,
    obj.task_object_insured_sum,
    obj.task_object_insured_sum_currency,
    obj.condition_min_insured_sum,
    obj.condition_max_insured_sum,
    case
        /* Не выбираем случайную СС, если в условиях есть расхождения. */
        when obj.condition_min_insured_sum =
             obj.condition_max_insured_sum
         and obj.condition_currency_count <= 1
        then obj.condition_max_insured_sum
    end as insured_sum,
    obj.insured_sum_currency,
    obj.condition_currency_count,
    obj.previous_contract_insured_sum,
    obj.previous_contract_amount_currency,
    obj.previous_object_insured_sum,
    obj.previous_object_insured_sum_currency,

    /* Премии, стоимости и лимиты. */
    obj.contract_premium,
    obj.previous_contract_premium,
    obj.insurance_value,
    obj.insurance_value_currency,
    obj.insurance_value_basis,
    obj.is_pledged,
    obj.pledged_value,
    obj.task_object_per_occurrence_limit,
    obj.minimum_per_occurrence_limit,
    obj.maximum_per_occurrence_limit,

    /* Простая договорная история. */
    (obj.previous_contract_id is not null) as has_previous_contract,
    obj.previous_contract_number,
    obj.previous_contract_start_date,
    obj.previous_contract_end_date,

    /* Исходные данные для проверки. */
    obj.condition_count,
    obj.conditions_json,
    obj.characteristics_version_number,
    obj.characteristics_version_start_date,
    obj.characteristics_version_end_date,
    obj.characteristics_version_is_active,
    obj.object_characteristics_json,
    obj.task_type,
    obj.task_status,
    obj.ins_refuse
from objects_with_previous obj
),

standardized_result as (
    /* Шаг 11. Приводим результат к общей структуре двух датасетов. */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    as_of_date desc nulls last,
    contract_id,
    object_id;

"""


In [ ]:
with engine.connect() as connection:
    strict_df = pd.read_sql_query(text(strict_sql), connection)

print('Строк:', len(strict_df))
print('Колонок:', len(strict_df.columns))
display(strict_df.head(3))


# 4. Проверка заполненности

In [ ]:
required_columns = {
    'contract_id', 'task_id', 'object_id', 'characteristics_id',
    'elementary_obj_type', 'insured_sum', 'full_address'
}
missing_columns = sorted(required_columns - set(strict_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных договоров',
        'Уникальных задач',
        'Уникальных объектов',
        'Уникальных пар задача + объект',
        'Строк с target',
        'Строк с адресом',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(strict_df),
        strict_df['contract_id'].nunique(dropna=True),
        strict_df['task_id'].nunique(dropna=True),
        strict_df['object_id'].nunique(dropna=True),
        strict_df[['task_id', 'object_id']].drop_duplicates().shape[0],
        strict_df['insured_sum'].notna().sum(),
        strict_df['full_address'].fillna('').str.strip().ne('').sum(),
        strict_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
duplicate_keys = (
    strict_df.groupby(['task_id', 'object_id'], dropna=False)
    .size()
    .gt(1)
    .sum()
)
print('Повторных ключей задача + объект:', duplicate_keys)
display(strict_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))

# 5. Поиск адресов страховых объектов в CDI

В CDI передаются только технический номер строки, CDI ID страхователя, ИНН, дата договора, адрес объекта и площадь. Одна строка CDI может вернуть несколько адресов-кандидатов. На этом этапе ничего не выбирается случайно.


In [ ]:
# готовим входные строки для CDI
sphere_with_row_id = strict_df.copy()
sphere_with_row_id.insert(
    0,
    'sphere_row_id',
    range(1, len(sphere_with_row_id) + 1),
)

sphere_with_row_id['source_address'] = (
    sphere_with_row_id['full_address']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
    .fillna(
        sphere_with_row_id['original_address']
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )
)

cdi_input_columns = [
    'sphere_row_id',
    'contract_id',
    'object_id',
    'policyholder_cdi_id',
    'policyholder_inn',
    'as_of_date',
    'source_address',
    'total_area',
]

missing_cdi_columns = [
    column for column in cdi_input_columns
    if column not in sphere_with_row_id.columns
]
if missing_cdi_columns:
    raise ValueError(
        'Для CDI не хватает колонок: ' + ', '.join(missing_cdi_columns)
    )

def json_scalar(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value)

cdi_records = []
for row in sphere_with_row_id[cdi_input_columns].itertuples(
    index=False,
    name=None,
):
    cdi_records.append({
        'sphere_row_id': int(row[0]),
        'contract_id': json_scalar(row[1]),
        'object_id': json_scalar(row[2]),
        'cdi_id': json_scalar(row[3]),
        'inn': json_scalar(row[4]),
        'as_of_date': json_scalar(row[5]),
        'source_address': json_scalar(row[6]),
        'total_area': json_scalar(row[7]),
    })

sphere_json = json.dumps(cdi_records, ensure_ascii=False)
print('Строк передано в CDI:', len(cdi_records))
print('Размер JSON, МБ:', round(len(sphere_json.encode('utf-8')) / 1024**2, 2))


In [ ]:
cdi_sql = r"""
/*
Запрос получает адреса страховых объектов из CDI.

Входные строки передаются из notebook в параметре sphere_json.
Запрос ничего не создаёт и не изменяет в КХД.

CDI ID относится к страхователю, а не к зданию. Поэтому запрос возвращает
всех действующих кандидатов с признаком адреса страхового объекта. Выбор
выполняется только после подсчёта клиентов и адресов.
*/

with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        s.contract_id,
        s.object_id,
        trim(s.cdi_id) as cdi_id,
        regexp_replace(s.inn, '[^0-9]', '') as inn,
        case
            when regexp_like(s.as_of_date, '^[0-9]{4}-[0-9]{2}-[0-9]{2}')
            then to_date(substr(s.as_of_date, 1, 10), 'YYYY-MM-DD')
        end as object_date,
        s.source_address,
        s.total_area
    from json_table(
        :sphere_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            contract_id varchar2(200) path '$.contract_id',
            object_id varchar2(200) path '$.object_id',
            cdi_id varchar2(500) path '$.cdi_id',
            inn varchar2(50) path '$.inn',
            as_of_date varchar2(100) path '$.as_of_date',
            source_address varchar2(4000) path '$.source_address',
            total_area varchar2(200) path '$.total_area'
        )
    ) s
),

resolved_clients_raw as (
    /* Прямой CLIENT_ID имеет первый приоритет. */
    select distinct
        s.sphere_row_id,
        trim(s.cdi_id) as khd_client_id,
        'DIRECT_CLIENT_ID' as client_match_method,
        1 as client_match_priority
    from sphere_objects s
    where s.cdi_id is not null
      and exists (
          select 1
          from DM_RISK_AVATAR.STG_ADDRESS_CDI_ZUD a
          where trim(a.client_id) = s.cdi_id
      )

    union all

    select distinct
        s.sphere_row_id,
        trim(p.party_hidparty) as khd_client_id,
        'PARTY_HIDPARTY' as client_match_method,
        2 as client_match_priority
    from sphere_objects s
    join DM_RISK_AVATAR.STG_PARTY_SRC p
        on trim(p.party_hidparty) = s.cdi_id
    where s.cdi_id is not null

    union all

    select distinct
        s.sphere_row_id,
        trim(p.party_hidparty) as khd_client_id,
        'PARTY_SOURCE_ID' as client_match_method,
        3 as client_match_priority
    from sphere_objects s
    join DM_RISK_AVATAR.STG_PARTY_SRC p
        on trim(p.source_id) = s.cdi_id
    where s.cdi_id is not null
      and p.party_hidparty is not null
),

resolved_clients as (
    select
        r.sphere_row_id,
        r.khd_client_id,
        min(r.client_match_method) keep (
            dense_rank first order by r.client_match_priority
        ) as client_match_method,
        min(r.client_match_priority) as client_match_priority
    from resolved_clients_raw r
    group by
        r.sphere_row_id,
        r.khd_client_id
),

client_checks as (
    select
        r.sphere_row_id,
        r.khd_client_id,
        r.client_match_method,
        r.client_match_priority,
        case
            when s.inn is null then 'not_checked'
            when exists (
                select 1
                from DM_RISK_AVATAR.STG_LEGAL_CLIENTS l
                where trim(l.client_id) = r.khd_client_id
                  and regexp_replace(l.inn, '[^0-9]', '') = s.inn
            ) then 'confirmed'
            when exists (
                select 1
                from DM_RISK_AVATAR.STG_LEGAL_CLIENTS l
                where trim(l.client_id) = r.khd_client_id
            ) then 'conflict'
            else 'not_checked'
        end as cdi_inn_status
    from resolved_clients r
    join sphere_objects s
        on s.sphere_row_id = r.sphere_row_id
),

client_summary as (
    select
        c.sphere_row_id,
        count(distinct c.khd_client_id) as resolved_client_count,
        min(c.client_match_method) as client_match_method,
        case
            when max(case when c.cdi_inn_status = 'confirmed' then 1 else 0 end) = 1
                then 'confirmed'
            when max(case when c.cdi_inn_status = 'conflict' then 1 else 0 end) = 1
                then 'conflict'
            else 'not_checked'
        end as cdi_inn_status
    from client_checks c
    group by c.sphere_row_id
),

address_candidates_raw as (
    select
        c.sphere_row_id,
        c.khd_client_id,
        c.client_match_method,
        c.client_match_priority,
        a.address_id,
        a.address_name,
        trim(a.fias_id_house) as fias_id_house,
        a.is_cdi,
        a.is_zud,
        a.insured_object_address_flag,
        a.insured_start_date,
        a.insured_end_date,
        a.verified_address_flag
    from client_checks c
    join sphere_objects s
        on s.sphere_row_id = c.sphere_row_id
    join DM_RISK_AVATAR.STG_ADDRESS_CDI_ZUD a
        on trim(a.client_id) = c.khd_client_id
    where a.insured_object_address_flag = 1
      and (
          s.object_date is null
          or (
              (a.insured_start_date is null
               or a.insured_start_date <= s.object_date)
              and (a.insured_end_date is null
                   or a.insured_end_date >= s.object_date)
          )
      )
),

address_candidates_ranked as (
    /* Нумеруем повторы одной адресной записи из разных путей к клиенту. */
    select
        a.*,
        row_number() over (
            partition by a.sphere_row_id, a.address_id
            order by
                a.client_match_priority,
                a.khd_client_id
        ) as address_row_number
    from address_candidates_raw a
),

address_candidates as (
    /* Оставляем одну запись адреса и не агрегируем его текстовое поле. */
    select a.*
    from address_candidates_ranked a
    where a.address_row_number = 1
),

result_rows as (
    select
        s.sphere_row_id,
        s.contract_id,
        s.object_id,
        s.cdi_id,
        s.inn,
        s.object_date,
        s.source_address,
        s.total_area,
        nvl(cs.resolved_client_count, 0) as resolved_client_count,
        cs.client_match_method,
        nvl(cs.cdi_inn_status, 'not_checked') as cdi_inn_status,
        a.address_id,
        a.address_name,
        a.fias_id_house,
        a.is_cdi,
        a.is_zud,
        a.insured_start_date,
        a.insured_end_date,
        a.verified_address_flag,
        count(a.address_id) over (
            partition by s.sphere_row_id
        ) as cdi_address_candidate_count,
        count(a.fias_id_house) over (
            partition by s.sphere_row_id
        ) as cdi_fias_candidate_count
    from sphere_objects s
    left join client_summary cs
        on cs.sphere_row_id = s.sphere_row_id
    left join address_candidates a
        on a.sphere_row_id = s.sphere_row_id
)

select /*+ no_parallel */
    r.sphere_row_id as "sphere_row_id",
    r.contract_id as "contract_id",
    r.object_id as "object_id",
    r.cdi_id as "cdi_id",
    r.inn as "inn",
    r.object_date as "object_date",
    r.source_address as "source_address",
    r.total_area as "total_area",
    r.resolved_client_count as "resolved_client_count",
    r.client_match_method as "cdi_client_match_method",
    r.cdi_inn_status as "cdi_inn_status",
    r.address_id as "cdi_address_id",
    r.address_name as "cdi_address_name",
    r.fias_id_house as "cdi_fias_id_house",
    r.is_cdi as "cdi_is_cdi",
    r.is_zud as "cdi_is_zud",
    r.verified_address_flag as "cdi_verified_address_flag",
    r.insured_start_date as "cdi_insured_start_date",
    r.insured_end_date as "cdi_insured_end_date",
    r.cdi_address_candidate_count as "cdi_address_candidate_count",
    r.cdi_fias_candidate_count as "cdi_fias_candidate_count"
from result_rows r
order by
    r.sphere_row_id,
    r.address_id

"""


In [ ]:
def safe_oracle_identifier(value, field_name):
    cleaned = value.replace('_', '')
    if not cleaned.isalnum():
        raise ValueError(f'Некорректное значение {field_name}')
    return value.upper()

khd_schema = safe_oracle_identifier(
    KHD_DATA_SCHEMA,
    'KHD_DATA_SCHEMA',
)

cdi_query = cdi_sql.replace(
    'DM_RISK_AVATAR.',
    f'{khd_schema}.',
)

with khd_connection.cursor() as cursor:
    sphere_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
    sphere_json_bind.setvalue(0, sphere_json)
    cursor.execute(cdi_query, sphere_json=sphere_json_bind)
    cdi_columns = [str(column[0]).lower() for column in cursor.description]
    cdi_rows = cursor.fetchall()

cdi_candidates_df = pd.DataFrame(cdi_rows, columns=cdi_columns)

required_cdi_result = {
    'sphere_row_id',
    'resolved_client_count',
    'cdi_inn_status',
    'cdi_address_id',
    'cdi_address_name',
    'cdi_fias_id_house',
    'cdi_address_candidate_count',
}
missing_cdi_result = sorted(
    required_cdi_result - set(cdi_candidates_df.columns)
)
if missing_cdi_result:
    raise ValueError(
        'CDI не вернул ожидаемые колонки: '
        + ', '.join(missing_cdi_result)
    )

print('Строк-кандидатов CDI:', len(cdi_candidates_df))
print(
    'Исходных строк с кандидатами:',
    cdi_candidates_df.loc[
        cdi_candidates_df['cdi_address_id'].notna(),
        'sphere_row_id',
    ].nunique(),
)


In [ ]:
# получаем одну служебную строку на каждый объект Сферы
cdi_base = (
    cdi_candidates_df
    .sort_values(['sphere_row_id', 'cdi_address_id'], na_position='last')
    .groupby('sphere_row_id', as_index=False)
    .first()
)

candidate_rows = cdi_candidates_df.loc[
    cdi_candidates_df['cdi_address_id'].notna()
].copy()

unique_candidates = candidate_rows.loc[
    candidate_rows['resolved_client_count'].eq(1)
    & candidate_rows['cdi_address_candidate_count'].eq(1)
    & candidate_rows['cdi_inn_status'].ne('conflict')
].copy()

if unique_candidates['sphere_row_id'].duplicated().any():
    raise ValueError('CDI вернул несколько выбранных адресов для одной строки')

selected_cdi_columns = [
    'sphere_row_id',
    'cdi_address_id',
    'cdi_address_name',
    'cdi_fias_id_house',
    'cdi_is_cdi',
    'cdi_is_zud',
    'cdi_verified_address_flag',
    'cdi_insured_start_date',
    'cdi_insured_end_date',
]

cdi_object_df = cdi_base[[
    'sphere_row_id',
    'cdi_id',
    'resolved_client_count',
    'cdi_client_match_method',
    'cdi_inn_status',
    'cdi_address_candidate_count',
    'cdi_fias_candidate_count',
]].merge(
    unique_candidates[selected_cdi_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

def cdi_status(row):
    if pd.isna(row['cdi_id']) or str(row['cdi_id']).strip() == '':
        return 'no_cdi_id'
    if row['resolved_client_count'] == 0:
        return 'client_not_found'
    if row['resolved_client_count'] > 1:
        return 'ambiguous_clients'
    if row['cdi_inn_status'] == 'conflict':
        return 'inn_conflict'
    if row['cdi_address_candidate_count'] == 0:
        return 'no_active_insured_address'
    if row['cdi_address_candidate_count'] > 1:
        return 'ambiguous_insured_addresses'
    if pd.isna(row['cdi_fias_id_house']):
        return 'unique_address_without_fias'
    return 'unique_active_insured_address'

cdi_object_df['cdi_match_status'] = cdi_object_df.apply(
    cdi_status,
    axis=1,
)
cdi_object_df['cdi_is_unique_match'] = (
    cdi_object_df['cdi_match_status']
    .eq('unique_active_insured_address')
    .astype('int64')
)

print('Статусы CDI:')
print(cdi_object_df['cdi_match_status'].value_counts(dropna=False))


# 6. Нормализация ФИАС через функцию CDI

Функция вызывается только для уникальных кодов ФИАС выбранных адресов. Повторные коды не запрашиваются заново внутри одного запуска. Ограничение — не более 20 вызовов в секунду, как в переданном примере.


In [ ]:
import time

function_schema = safe_oracle_identifier(
    CDI_FUNCTION_SCHEMA,
    'CDI_FUNCTION_SCHEMA',
)
function_name = safe_oracle_identifier(
    CDI_FUNCTION_NAME,
    'CDI_FUNCTION_NAME',
)

fias_codes = (
    cdi_object_df.loc[
        cdi_object_df['cdi_is_unique_match'].eq(1),
        'cdi_fias_id_house',
    ]
    .dropna()
    .astype('string')
    .str.strip()
    .loc[lambda values: values.ne('')]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

cdi_lookup_sql = f"""
select
    FEDERAL_DISTRICT,
    REGION_FIAS_ID,
    REGION_WITH_TYPE,
    AREA_FIAS_ID,
    AREA_WITH_TYPE,
    CITY_FIAS_ID,
    CITY,
    CITY_AREA,
    CITY_DISTRICT_FIAS_ID,
    CITY_DISTRICT,
    SETTLEMENT_FIAS_ID,
    SETTLEMENT,
    STREET_FIAS_ID,
    FLAT_FIAS_ID,
    HOUSE_FIAS_ID,
    FIAS_ID,
    FIAS_LEVEL,
    GEO_LAT,
    GEO_LON
from {function_schema}.{function_name}(:fias_code)
"""

lookup_rows = []
batch_started = time.monotonic()
for number, fias_code in enumerate(fias_codes, start=1):
    with khd_connection.cursor() as cursor:
        cursor.execute(cdi_lookup_sql, fias_code=fias_code)
        columns = [str(column[0]).lower() for column in cursor.description]
        rows = cursor.fetchall()

    for row in rows:
        record = dict(zip(columns, row))
        record['input_fias_id_house'] = fias_code
        lookup_rows.append(record)

    if number % 20 == 0:
        elapsed = time.monotonic() - batch_started
        if elapsed < 1:
            time.sleep(1 - elapsed)
        batch_started = time.monotonic()
    if number % 100 == 0:
        print('Обработано уникальных ФИАС:', number)

cdi_fias_lookup_df = pd.DataFrame(lookup_rows)

lookup_summary_rows = []
for fias_code in fias_codes:
    if cdi_fias_lookup_df.empty:
        rows_for_code = pd.DataFrame()
    else:
        rows_for_code = cdi_fias_lookup_df.loc[
            cdi_fias_lookup_df['input_fias_id_house'].eq(fias_code)
        ]

    if rows_for_code.empty:
        house_codes = []
    else:
        house_codes = (
            rows_for_code['house_fias_id']
            .dropna()
            .astype('string')
            .str.strip()
            .loc[lambda values: values.ne('')]
            .drop_duplicates()
            .tolist()
        )

    if len(house_codes) > 1:
        lookup_status = 'ambiguous_house_fias'
        normalized_house_fias = None
    elif len(house_codes) == 1:
        lookup_status = 'normalized'
        normalized_house_fias = house_codes[0]
    else:
        lookup_status = 'input_fias_used'
        normalized_house_fias = fias_code

    first_row = rows_for_code.iloc[0] if not rows_for_code.empty else None
    lookup_summary_rows.append({
        'cdi_fias_id_house': fias_code,
        'cdi_lookup_status': lookup_status,
        'cdi_lookup_row_count': len(rows_for_code),
        'cdi_normalized_house_fias_id': normalized_house_fias,
        'cdi_fias_level': None if first_row is None else first_row.get('fias_level'),
        'cdi_geo_lat': None if first_row is None else first_row.get('geo_lat'),
        'cdi_geo_lon': None if first_row is None else first_row.get('geo_lon'),
    })

cdi_fias_summary_df = pd.DataFrame(lookup_summary_rows)
if cdi_fias_summary_df.empty:
    cdi_fias_summary_df = pd.DataFrame(columns=[
        'cdi_fias_id_house',
        'cdi_lookup_status',
        'cdi_lookup_row_count',
        'cdi_normalized_house_fias_id',
        'cdi_fias_level',
        'cdi_geo_lat',
        'cdi_geo_lon',
    ])

cdi_object_df = cdi_object_df.merge(
    cdi_fias_summary_df,
    on='cdi_fias_id_house',
    how='left',
    validate='many_to_one',
)
cdi_object_df['egrn_fias_id_house'] = (
    cdi_object_df['cdi_normalized_house_fias_id']
    .where(cdi_object_df['cdi_is_unique_match'].eq(1))
)

print('Уникальных ФИАС передано функции CDI:', len(fias_codes))
print('Строк с ФИАС для поиска ЕГРН:', cdi_object_df['egrn_fias_id_house'].notna().sum())


# 7. Поиск здания в ЕГРН по ФИАС дома

ФИАС дома может соответствовать нескольким кадастровым объектам. Запрос оставляет только здания, сооружения и строения и удаляет повторные версии записей.

Если найден один объект, он присоединяется без проверки площади. Если объектов несколько и площадь Сферы заполнена, площадь используется для уточнения. Если площадь отсутствует или после сравнения всё равно остаётся несколько вариантов, данные ЕГРН не присоединяются. Исходная строка объекта при этом не удаляется.


In [ ]:
egrn_input = cdi_object_df.loc[
    cdi_object_df['egrn_fias_id_house'].notna(),
    ['sphere_row_id', 'egrn_fias_id_house'],
].merge(
    sphere_with_row_id[['sphere_row_id', 'total_area']],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

egrn_records = [
    {
        'sphere_row_id': int(row.sphere_row_id),
        'fias_id_house': json_scalar(row.egrn_fias_id_house),
        'total_area': json_scalar(row.total_area),
    }
    for row in egrn_input.itertuples(index=False)
]
egrn_json = json.dumps(egrn_records, ensure_ascii=False)

print('Строк передано в поиск ЕГРН:', len(egrn_records))


In [ ]:
egrn_by_fias_sql = r"""
/*
Запрос ищет здания ЕГРН по ФИАС дома, полученному из CDI.

ФИАС дома используется только для формирования кандидатов. Если найдено
несколько кадастровых зданий, дополнительно проверяется площадь. Данные ЕГРН
возвращаются только при одном кандидате после проверки.
*/

with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        trim(s.fias_id_house) as fias_id_house,
        replace(
            regexp_replace(trim(s.total_area), '[[:space:]]+', ''),
            ',',
            '.'
        ) as sphere_area_text
    from json_table(
        :egrn_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            fias_id_house varchar2(500) path '$.fias_id_house',
            total_area varchar2(200) path '$.total_area'
        )
    ) s
),

sphere_prepared as (
    select
        s.*,
        case
            when regexp_like(s.sphere_area_text, '^[0-9]+([.][0-9]+)?$')
            then to_number(
                s.sphere_area_text,
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area
    from sphere_objects s
),

egrn_raw as (
    select /*+ no_parallel(e) */
        s.sphere_row_id,
        s.sphere_area,
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || cast(e.cad_ind as varchar2(200))
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        case
            when regexp_like(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as egrn_area,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date
    from sphere_prepared s
    join DM_RISK_AVATAR.EGRN_DATA e
        on e.fias_id_house = s.fias_id_house
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) in (
          'здание',
          'сооружение',
          'строение'
      )
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (e.cadaster is not null or e.cad_ind is not null)
),

ranked_egrn as (
    select
        e.*,
        row_number() over (
            partition by e.sphere_row_id, e.egrn_key
            order by
                e.row_update_date desc nulls last,
                e.ias_update_date desc nulls last,
                e.cad_ind desc nulls last
        ) as version_number
    from egrn_raw e
),

one_row_per_object as (
    select e.*
    from ranked_egrn e
    where e.version_number = 1
),

area_check as (
    select
        e.*,
        count(*) over (
            partition by e.sphere_row_id
        ) as address_candidate_count,
        case
            when e.sphere_area > 0
             and e.egrn_area is not null
             and abs(e.egrn_area - e.sphere_area)
                 <= greatest(1, e.sphere_area * 0.01)
                then 1
            else 0
        end as area_matches
    from one_row_per_object e
),

area_choice as (
    select
        e.*,
        max(e.area_matches) over (
            partition by e.sphere_row_id
        ) as has_area_match
    from area_check e
),

candidates_after_area as (
    select e.*
    from area_choice e
    where e.has_area_match = 0
       or e.area_matches = 1
),

candidate_counts as (
    select
        e.*,
        count(*) over (
            partition by e.sphere_row_id
        ) as candidate_count
    from candidates_after_area e
),

candidate_summary as (
    select
        e.sphere_row_id,
        max(e.address_candidate_count) as address_candidate_count,
        max(e.candidate_count) as candidate_count,
        max(e.has_area_match) as has_area_match
    from candidate_counts e
    group by e.sphere_row_id
),

chosen_egrn as (
    select e.*
    from candidate_counts e
    where e.candidate_count = 1
)

select /*+ no_parallel */
    s.sphere_row_id as "sphere_row_id",
    s.fias_id_house as "cdi_fias_id_house",
    nvl(cs.address_candidate_count, 0) as "egrn_address_candidate_count",
    nvl(cs.candidate_count, 0) as "egrn_candidate_count",
    case
        when cs.candidate_count = 1 then 1
        else 0
    end as "egrn_is_unique_match",
    case
        when nvl(cs.candidate_count, 0) = 0
            then 'not_found'
        when cs.candidate_count > 1
            then 'ambiguous'
        when cs.has_area_match = 1
            then 'fias_house_and_area'
        else 'fias_house_only'
    end as "egrn_match_method",
    e.cad_ind as "egrn_cad_ind",
    e.cadaster as "egrn_cadaster",
    e.egrn_address as "egrn_address",
    e.square as "egrn_square",
    e.measure as "egrn_measure",
    e.building_type as "egrn_building_type",
    e.oks_type as "egrn_oks_type",
    e.oks_purpose as "egrn_oks_purpose",
    e.object_status as "egrn_object_status",
    e.fias_level as "egrn_fias_level",
    e.fias_id_house as "egrn_fias_id_house"
from sphere_prepared s
left join candidate_summary cs
    on cs.sphere_row_id = s.sphere_row_id
left join chosen_egrn e
    on e.sphere_row_id = s.sphere_row_id
order by s.sphere_row_id


"""


In [ ]:
egrn_expected_columns = [
    'sphere_row_id',
    'cdi_fias_id_house',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_address',
    'egrn_square',
    'egrn_measure',
    'egrn_building_type',
    'egrn_oks_type',
    'egrn_oks_purpose',
    'egrn_object_status',
    'egrn_fias_level',
    'egrn_fias_id_house',
]

if egrn_records:
    egrn_query = egrn_by_fias_sql.replace(
        'DM_RISK_AVATAR.',
        f'{khd_schema}.',
    )
    with khd_connection.cursor() as cursor:
        egrn_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
        egrn_json_bind.setvalue(0, egrn_json)
        cursor.execute(egrn_query, egrn_json=egrn_json_bind)
        egrn_columns = [
            str(column[0]).lower()
            for column in cursor.description
        ]
        egrn_rows = cursor.fetchall()
    egrn_lookup_df = pd.DataFrame(egrn_rows, columns=egrn_columns)
else:
    egrn_lookup_df = pd.DataFrame(columns=egrn_expected_columns)

missing_egrn_columns = sorted(
    set(egrn_expected_columns) - set(egrn_lookup_df.columns)
)
if missing_egrn_columns:
    raise ValueError(
        'ЕГРН не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('ЕГРН вернул несколько итоговых строк для объекта Сферы')

cdi_egrn_df = cdi_object_df.merge(
    egrn_lookup_df[egrn_expected_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
    suffixes=('', '_egrn_result'),
)

cdi_egrn_df['egrn_is_unique_match'] = (
    cdi_egrn_df['egrn_is_unique_match']
    .fillna(0)
    .astype('int64')
)

cdi_columns_for_result = [
    column for column in cdi_egrn_df.columns
    if column != 'cdi_id'
]

strict_cdi_egrn_df = sphere_with_row_id.merge(
    cdi_egrn_df[cdi_columns_for_result],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

strict_cdi_egrn_df['pipeline_match_status'] = 'not_linked'
strict_cdi_egrn_df.loc[
    strict_cdi_egrn_df['cdi_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'cdi_linked'
strict_cdi_egrn_df.loc[
    strict_cdi_egrn_df['egrn_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'egrn_linked'

if len(strict_cdi_egrn_df) != len(strict_df):
    raise ValueError('После CDI и ЕГРН изменилось количество строк датасета')

print('Строк в итоговом датасете:', len(strict_cdi_egrn_df))


# 8. Контроль результата

Проверки выводят только агрегаты. Реальные адреса, ИНН и номера договоров на экран не выводятся.


In [ ]:
quality_profile = pd.DataFrame({
    'Показатель': [
        'Строк исходного строгого датасета',
        'Строк итогового датасета',
        'Уникальных объектов',
        'Строк с CDI ID',
        'Однозначных адресов CDI с ФИАС',
        'Однозначных зданий ЕГРН',
        'Неоднозначных адресов CDI',
        'Неоднозначных зданий ЕГРН',
    ],
    'Значение': [
        len(strict_df),
        len(strict_cdi_egrn_df),
        strict_cdi_egrn_df['object_id'].nunique(dropna=True),
        strict_cdi_egrn_df['policyholder_cdi_id'].notna().sum(),
        strict_cdi_egrn_df['cdi_is_unique_match'].fillna(0).sum(),
        strict_cdi_egrn_df['egrn_is_unique_match'].fillna(0).sum(),
        strict_cdi_egrn_df['cdi_match_status']
            .eq('ambiguous_insured_addresses').sum(),
        strict_cdi_egrn_df['egrn_match_method']
            .eq('ambiguous').sum(),
    ],
})
display(quality_profile)

print()
print('Статусы CDI:')
print(strict_cdi_egrn_df['cdi_match_status'].value_counts(dropna=False))
print()
print('Методы ЕГРН:')
print(strict_cdi_egrn_df['egrn_match_method'].value_counts(dropna=False))


# 9. Сохранение локальных снимков и итогового датасета


In [ ]:
snapshot_at = pd.Timestamp.now(tz='Europe/Moscow').isoformat()
strict_cdi_egrn_df['external_snapshot_at'] = snapshot_at

final_path = OUTPUT_DIR / 'датасет_строгий_CDI_ЕГРН.csv'
cdi_candidates_path = OUTPUT_DIR / 'снимок_CDI_кандидаты.csv'
cdi_fias_path = OUTPUT_DIR / 'снимок_CDI_ФИАС.csv'
egrn_path = OUTPUT_DIR / 'снимок_ЕГРН_по_CDI.csv'

strict_cdi_egrn_df.to_csv(
    final_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)
cdi_candidates_df.assign(snapshot_at=snapshot_at).to_csv(
    cdi_candidates_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)
cdi_fias_lookup_df.assign(snapshot_at=snapshot_at).to_csv(
    cdi_fias_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)
egrn_lookup_df.assign(snapshot_at=snapshot_at).to_csv(
    egrn_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

print('Итоговый датасет:', final_path)
print('Снимок кандидатов CDI:', cdi_candidates_path)
print('Снимок нормализации CDI:', cdi_fias_path)
print('Снимок ЕГРН:', egrn_path)


# 10. Список уникальных ИНН

Пустые значения и повторы исключаются.


In [ ]:
inn_df = (
    strict_cdi_egrn_df[['policyholder_inn']]
    .rename(columns={'policyholder_inn': 'inn'})
    .assign(inn=lambda data: data['inn'].astype('string').str.strip())
    .loc[lambda data: data['inn'].notna() & data['inn'].ne('')]
    .drop_duplicates(subset=['inn'])
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(inn_path, index=False, sep=';', encoding='utf-8-sig')
print('Уникальных ИНН:', len(inn_df))
print('Сохранено:', inn_path)


In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')
